# 팀 단위 as-of 피처 실험 노트북

`train_ensemble.py`에 새로 추가한 `build_team_features`(팀 단위 as-of 성공률,
`pitcher_team_id`/`batter_team_id` 기준)가 LightGBM 성능을 개선하는지 확인합니다.

**사전 검증 결과** (구현 전에 확인함):
- 팀 ID는 13개뿐이라 팀당 평균 11만행 — 콜드스타트 비율 0.001%로 사실상 무시 가능
- `pitcher_team_id`는 시즌 중 이적으로 바뀔 수 있음(선수의 20.7%가 커리어 중 팀ID 2개 이상)
  → team_id로 그룹을 나누므로 이적 시점에 자동으로 전환되어 별도 처리 불필요
- 상관계수: `asof_pitcher_team_success_rate` 0.0422, `asof_batter_team_success_rate` 0.0380
  (참고 - 개인 단위 `asof_pitcher_success_rate`는 0.0843) → 개인 단위보다 신호가 약함
  (팀 단위로 뭉치면서 개인차가 희석되는 건 당연하지만, 그만큼 노이즈도 적음)

**비교 기준선** (좌우 상성 피처 채택 후 갱신된 값, `feature_platoon.ipynb` 참고):
- LightGBM 전체(147만행) OOF Brier: **0.24405**

**이 파일과 `train_ensemble.py`는 같은 폴더에 있어야 아래 import가 동작합니다.**

In [1]:
import sys, os, time
import numpy as np
import pandas as pd
from sklearn.metrics import brier_score_loss

sys.path.append(os.getcwd())
from train_ensemble import (
    TARGET_COL, CAT_COLS, build_features, train_lgb,
)

DATA_DIR = "../open/data"

## 1. 데이터 로드 & 피처 생성 (팀 단위 피처 포함)

`use_team_feature=True`로 켜야 팀 단위 피처가 포함됩니다(기본값 False,
아직 검증 전이라 다른 노트북에 영향 없도록 안전하게 꺼둔 상태).

In [2]:
train = pd.read_csv(os.path.join(DATA_DIR, "train.csv"), encoding="utf-8-sig")
print(train.shape)

train_feat, feat_cols = build_features(train, None, use_team_feature=True)
cat_features = [c for c in CAT_COLS if c in feat_cols]

team_cols = [c for c in feat_cols if "team" in c and c.startswith("asof")]
print(f"피처 개수: {len(feat_cols)} (팀 단위 피처: {team_cols})")

r = train_feat[TARGET_COL].mean()
baseline_brier = r * (1 - r)
print(f"기준(무정보) Brier = {baseline_brier:.5f}")

(1475092, 49)
피처 개수: 75 (팀 단위 피처: ['asof_pitcher_team_n', 'asof_pitcher_team_success_rate', 'asof_batter_team_n', 'asof_batter_team_success_rate'])
기준(무정보) Brier = 0.24944


## 2. 전체 데이터로 바로 확인

지금까지의 경험상(맞대결 피처는 20만행-전체 결과 부호가 뒤집혔고, 좌우 상성 피처는
20만행-전체가 같은 방향이었음) 20만행 결과의 신뢰도가 들쭉날쭉하므로, 이번엔
20만행 단계를 생략하고 바로 전체 데이터로 확인합니다.

In [3]:
X_full = train_feat[feat_cols]
y_full = train_feat[TARGET_COL].values

t0 = time.time()
lgb_models_team_full, lgb_oof_team_full = train_lgb(X_full, y_full, X_full, cat_features)
brier_team_full = brier_score_loss(y_full, lgb_oof_team_full)
print(f"[LightGBM+팀피처] 소요시간: {time.time()-t0:.1f}초")
print(f"[LightGBM+팀피처] OOF Brier (전체): {brier_team_full:.5f}")
print(f"참고 - 팀 피처 없는 LightGBM 전체 Brier(상성 피처 포함): 0.24405")
print(f"개선폭: {(0.24405 - brier_team_full) / 0.24405 * 100:.4f}% (양수면 개선)")

  [LGB fold 0] brier=0.24378
  [LGB fold 1] brier=0.24386
  [LGB fold 2] brier=0.24373
  [LGB fold 3] brier=0.24389
  [LGB fold 4] brier=0.24383
[LightGBM+팀피처] 소요시간: 229.6초
[LightGBM+팀피처] OOF Brier (전체): 0.24382
참고 - 팀 피처 없는 LightGBM 전체 Brier(상성 피처 포함): 0.24405
개선폭: 0.0956% (양수면 개선)


## 3. Feature Importance로 실제 활용도 확인

In [4]:
imp_df = pd.DataFrame({
    f"fold{i}": m.feature_importance(importance_type="gain")
    for i, m in enumerate(lgb_models_team_full)
}, index=feat_cols)
imp_df["mean_gain"] = imp_df[[c for c in imp_df.columns if c.startswith("fold")]].mean(axis=1)
imp_df["share_pct"] = imp_df["mean_gain"] / imp_df["mean_gain"].sum() * 100
imp_df = imp_df.sort_values("mean_gain", ascending=False)

imp_df_ranked = imp_df.reset_index().rename(columns={"index": "feature"})
imp_df_ranked["rank"] = imp_df_ranked.index + 1
print("팀 단위 피처 순위:")
display(imp_df_ranked[imp_df_ranked["feature"].isin(team_cols)][["rank", "feature", "mean_gain", "share_pct"]])

팀 단위 피처 순위:


,rank,feature,mean_gain,share_pct
2,3,asof_pitcher_team_n,43393.047976,3.823979
6,7,asof_batter_team_n,36885.770650,3.250530
7,8,asof_batter_team_success_rate,34241.140692,3.017474
12,13,asof_pitcher_team_success_rate,29420.971888,2.592700
